# 서울 랜드마크 이미지 분류


---
# ===== A 파트 (연주) =====
### 데이터 로드 / 경로 생성 / Dataset / DataLoader / 전처리

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch.nn as nn
import torch.optim as optim

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split


In [3]:
# GPU 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 중인 디바이스: {device}')

사용 중인 디바이스: cpu


## 1. 경로 설정 및 CSV 불러오기

> `train.csv`에는 파일명(`001.PNG`)과 라벨(0~9)이 있어.
> 실제 이미지를 불러오려면 파일명 앞에 폴더 경로를 붙여줘야 함

In [10]:
# 기본 경로 설정
BASE_DIR = '/content/drive/MyDrive/synaps_team_project'
TRAIN_DIR = os.path.join(BASE_DIR, 'train')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

# CSV 불러오기
train_df = pd.read_csv(os.path.join(BASE_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(BASE_DIR, 'test.csv'))
sample_submission = pd.read_csv(os.path.join(BASE_DIR, 'sample_submission.csv'))

print('=== train.csv ===')
print(train_df.head())
print(f'\n총 학습 데이터: {len(train_df)}개')
print(f'클래스 종류: {sorted(train_df["label"].unique())}')
print(f'클래스 수: {train_df["label"].nunique()}개')
print('\n=== test.csv ===')
print(test_df.head())
print(f'\n총 테스트 데이터: {len(test_df)}개')

=== train.csv ===
  file_name  label
0   001.PNG      9
1   002.PNG      4
2   003.PNG      1
3   004.PNG      1
4   005.PNG      6

총 학습 데이터: 723개
클래스 종류: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
클래스 수: 10개

=== test.csv ===
  file_name
0   001.PNG
1   002.PNG
2   003.PNG
3   004.PNG
4   005.PNG

총 테스트 데이터: 199개


## 2. 이미지 파일 경로 생성

> 파일명에 폴더 경로를 붙여서 실제로 이미지를 열 수 있는 전체 경로를 만들어.

In [11]:
# 파일명 → 전체 경로로 변환
train_df['file_path'] = train_df['file_name'].apply(lambda x: os.path.join(TRAIN_DIR, x))
test_df['file_path']  = test_df['file_name'].apply(lambda x: os.path.join(TEST_DIR, x))

print('경로 생성 완료')
print(train_df[['file_name', 'file_path', 'label']].head())

경로 생성 완료
  file_name                                          file_path  label
0   001.PNG  /content/drive/MyDrive/synaps_team_project/tra...      9
1   002.PNG  /content/drive/MyDrive/synaps_team_project/tra...      4
2   003.PNG  /content/drive/MyDrive/synaps_team_project/tra...      1
3   004.PNG  /content/drive/MyDrive/synaps_team_project/tra...      1
4   005.PNG  /content/drive/MyDrive/synaps_team_project/tra...      6


## 3. 학습 / 검증 데이터 분리

> 학습: 검증데이터 = 8:2로 나눠서 검증 데이터로 중간 확인을 해!
> - train: 578개
> - validation: 145개

In [12]:
# 8:2 비율로 분리 (random_state=42 → 항상 같은 방식으로 나뉘게)
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label']  # 클래스 비율 유지
)

# 인덱스 초기화
train_data = train_data.reset_index(drop=True)
val_data   = val_data.reset_index(drop=True)

print(f'학습 데이터: {len(train_data)}개')
print(f'검증 데이터: {len(val_data)}개')

학습 데이터: 578개
검증 데이터: 145개


## 4. 이미지 전처리 (Transform)

> 이미지를 그대로 모델에 넣으면 안 되고 통일된 형태로 변환해야 해!
> - **Resize**: 모든 이미지를 224x224로 통일
> - **ToTensor**: 이미지를 숫자 배열(텐서)로 변환
> - **Normalize**: 픽셀값 범위를 줄여서 학습 안정화
>
> 학습용과 검증용 전처리를 다르게 하는 이유:
> → 학습할 때만 데이터 증강(뒤집기, 회전 등)을 적용해서 모델을 더 강하게 만들어!

In [13]:
# ImageNet 기준 평균/표준편차 (전이학습에서도 동일하게 사용)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# 학습용 전처리 (데이터 증강 포함)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),   # 좌우 뒤집기
    transforms.RandomRotation(10),       # ±10도 회전
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

# 검증/테스트용 전처리 (증강 없이 그대로)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

print('전처리 설정 완료')

전처리 설정 완료


## 5. CustomDataset 구성

> PyTorch는 데이터를 불러올 때 정해진 형식이 있어.
> `__len__`: 데이터 개수 반환
> `__getitem__`: index를 주면 해당 이미지 + label 반환

In [14]:
class CustomDataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        """
        df       : 파일 경로와 라벨이 담긴 DataFrame
        transform: 이미지 전처리
        is_test  : test 데이터면 True (라벨 없음)
        """
        self.df        = df
        self.transform = transform
        self.is_test   = is_test

    def __len__(self):
        # 데이터 총 개수 반환
        return len(self.df)

    def __getitem__(self, idx):
        # 이미지 경로에서 이미지 불러오기
        img_path = self.df.loc[idx, 'file_path']
        image    = Image.open(img_path).convert('RGB')  # RGB로 통일

        # 전처리 적용
        if self.transform:
            image = self.transform(image)

        # test 데이터는 라벨 없음
        if self.is_test:
            return image

        label = self.df.loc[idx, 'label']
        return image, label

print('CustomDataset 정의 완료')

CustomDataset 정의 완료


## 6. DataLoader 구성

> Dataset이 창고라면 DataLoader는 배달부야!
> 이미지를 batch_size 단위로 묶어서 모델에 전달해.
> - `batch_size=32`: 32개씩 묶어서 전달
> - `shuffle=True`: 학습할 때마다 순서 섞기 (과적합 방지)

In [15]:
BATCH_SIZE = 32

# Dataset 생성
train_dataset = CustomDataset(train_data, transform=train_transform)
val_dataset   = CustomDataset(val_data,   transform=val_transform)
test_dataset  = CustomDataset(test_df,    transform=val_transform, is_test=True)

# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'train_loader: {len(train_loader)}개 배치')
print(f'val_loader:   {len(val_loader)}개 배치')
print(f'test_loader:  {len(test_loader)}개 배치')

# 정상 동작 확인
images, labels = next(iter(train_loader))
print(f'\n배치 이미지 shape: {images.shape}')  # (32, 3, 224, 224)
print(f'배치 라벨 shape:   {labels.shape}')   # (32,)

train_loader: 19개 배치
val_loader:   5개 배치
test_loader:  7개 배치

배치 이미지 shape: torch.Size([32, 3, 224, 224])
배치 라벨 shape:   torch.Size([32])


---
# ===== B 파트 (민진) =====
### CNN 모델 정의 / 학습 루프 / 검증 / test 예측 / submission 생성


## 1. CNN 모델 정의

> CNN 모델은 이미지를 입력받아 특징을 추출하고 클래스를 분류하는 딥러닝 모델 구조
> - Conv2d: 이미지 특징 추출
> - ReLU: 비선형성 추가
> - MaxPool2d: 특징 압축 및 차원 축소
> - Linear: 최종 클래스 분류
> - Dropout: 과적합 방지

In [16]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()

        self.features = nn.Sequential(
            # 입력: [batch, 3, 224, 224]
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 224 -> 112

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 112 -> 56

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 56 -> 28
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = SimpleCNN(num_classes=10).to(device)

print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=100352, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


2. Loss 함수 / Optimizer 설정

> Loss 함수는 모델의 예측값과 실제 정답의 차이를 계산하는 함수

> - CrossEntropyLoss: 다중 클래스 이미지 분류 문제에서 사용하는 손실 함수

> Optimizer는 loss 값을 줄이도록 모델 가중치를 업데이트하는 알고리즘

> - Adam: 학습 속도와 안정성을 함께 고려한 최적화 기법
> - lr=0.001: 가중치 업데이트 크기를 조절하는 학습률 설정

In [17]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss 함수와 Optimizer 설정 완료")

Loss 함수와 Optimizer 설정 완료


3. 학습 함수 정의

> 학습 함수는 train 데이터를 이용하여 CNN 모델을 반복 학습하는 과정

> - model.train(): 학습 모드 설정
> - forward: 입력 이미지를 모델에 전달하여 예측 수행
> - loss 계산: 예측값과 실제 정답 차이 계산
> - backward: 오차 역전파 수행
> - optimizer.step(): 가중치 업데이트
> - accuracy 계산: 모델 분류 성능 확인

In [18]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        # gradient 초기화
        optimizer.zero_grad()

        # 예측
        outputs = model(images)

        # loss 계산
        loss = criterion(outputs, labels)

        # 역전파
        loss.backward()

        # 가중치 업데이트
        optimizer.step()

        # loss 누적
        running_loss += loss.item()

        # accuracy 계산
        _, predicted = outputs.max(1)

        total += labels.size(0)

        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


print("학습 함수 정의 완료")

학습 함수 정의 완료


4. 검증 함수 정의

> 검증 함수는 validation 데이터를 이용하여 학습된 모델 성능을 평가하는 과정

> - model.eval(): 검증 모드 설정
> - torch.no_grad(): 불필요한 gradient 계산 비활성화
> - forward: validation 데이터 예측 수행
> - loss 계산: validation loss 확인
> - accuracy 계산: validation accuracy 측정
> - 과적합 여부 및 일반화 성능 확인

In [19]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    # 검증에서는 gradient 계산 안 함
    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            # 예측
            outputs = model(images)

            # loss 계산
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            # accuracy 계산
            _, predicted = outputs.max(1)

            total += labels.size(0)

            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


print("검증 함수 정의 완료")

검증 함수 정의 완료


5. 학습 실행

> Epoch 단위로 학습 함수와 검증 함수를 반복 수행하는 과정

> - train_one_epoch(): train 데이터 학습 수행
> - validate(): validation 데이터 성능 평가
> - Train Loss: 학습 데이터 오차 확인
> - Train Accuracy: 학습 데이터 정확도 확인
> - Val Loss: validation 데이터 오차 확인
> - Val Accuracy: validation 데이터 정확도 확인
> - epoch별 성능 변화를 기록하여 학습 상태 분석

In [20]:
EPOCHS = 10

train_losses = []
train_accs = []

val_losses = []
val_accs = []

for epoch in range(EPOCHS):

    # train
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    # validation
    val_loss, val_acc = validate(
        model,
        val_loader,
        criterion,
        device
    )

    # 기록 저장
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f'Epoch [{epoch+1}/{EPOCHS}]')
    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
    print('-' * 50)

Epoch [1/10]
Train Loss: 2.7657 | Train Acc: 0.1332
Val Loss: 2.1053 | Val Acc: 0.2897
--------------------------------------------------
Epoch [2/10]
Train Loss: 1.8838 | Train Acc: 0.3564
Val Loss: 1.3879 | Val Acc: 0.5310
--------------------------------------------------
Epoch [3/10]
Train Loss: 1.2772 | Train Acc: 0.5744
Val Loss: 1.0166 | Val Acc: 0.6897
--------------------------------------------------
Epoch [4/10]
Train Loss: 0.9959 | Train Acc: 0.6817
Val Loss: 0.7840 | Val Acc: 0.7379
--------------------------------------------------
Epoch [5/10]
Train Loss: 0.8192 | Train Acc: 0.7353
Val Loss: 0.7192 | Val Acc: 0.7310
--------------------------------------------------
Epoch [6/10]
Train Loss: 0.7149 | Train Acc: 0.7751
Val Loss: 0.5810 | Val Acc: 0.8276
--------------------------------------------------
Epoch [7/10]
Train Loss: 0.7692 | Train Acc: 0.7820
Val Loss: 0.6141 | Val Acc: 0.7724
--------------------------------------------------
Epoch [8/10]
Train Loss: 0.5870 | 

6. test 이미지 예측

> 학습된 CNN 모델을 이용하여 정답이 없는 test 이미지의 클래스를 예측하는 과정

> - model.eval(): 예측 모드 설정
> - torch.no_grad(): gradient 계산 비활성화
> - test_loader: test 이미지를 batch 단위로 모델에 전달
> - outputs.max(1): 가장 높은 확률을 가진 클래스를 예측값으로 선택
> - predictions: test 이미지별 예측 label 저장

In [21]:
model.eval()

predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)

        outputs = model(images)
        _, predicted = outputs.max(1)

        predictions.extend(predicted.cpu().numpy())

print("test 예측 완료")
print(predictions[:10])
print(f"총 예측 개수: {len(predictions)}")

test 예측 완료
[np.int64(7), np.int64(1), np.int64(9), np.int64(7), np.int64(6), np.int64(1), np.int64(3), np.int64(6), np.int64(3), np.int64(1)]
총 예측 개수: 199


7. submission 파일 생성

> test 이미지 예측 결과를 제출 양식에 맞게 CSV 파일로 저장하는 과정

> - sample_submission.copy(): 제공된 제출 양식 복사
> - submission['label'] = predictions: 예측 label 입력
> - to_csv(): 최종 제출 파일 저장
> - submission_cnn.csv: 기본 CNN 모델의 test 예측 결과 파일

In [22]:
submission = sample_submission.copy()

submission['label'] = predictions

submission.to_csv(
    os.path.join(BASE_DIR, 'submission_cnn.csv'),
    index=False
)

print("submission_cnn.csv 저장 완료")
print(submission.head())

submission_cnn.csv 저장 완료
  file_name  label
0   001.PNG      7
1   002.PNG      1
2   003.PNG      9
3   004.PNG      7
4   005.PNG      6
